# ComfyUI on Google Colab（多底模 + LoRA 通用生图启动器）

支持 **Z-Image-Turbo（GGUF / FP8 / BF16）**、**SD 1.5**、**SDXL / Pony / Illustrious** 等底模与 LoRA，启动后通过 `cloudflared` 直接打开 ComfyUI 网页版。

### 使用顺序
1. 在顶部菜单选择 **代码执行程序 → 更改运行时类型 → GPU（T4 或 L4）**
2. 按顺序运行 **第 1 步 → 第 5 步**
3. 第 5 步输出 `https://xxxx.trycloudflare.com` 后，点击即可进入 ComfyUI 网页版（内置预设工作流已在菜单 `Workflows` 中）

In [ ]:
#@title 1. 检查 GPU 显存环境 & 可选挂载 Google Drive
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_WORK_DIR = "/content/drive/MyDrive/ComfyUI_Colab"  #@param {type:"string"}

import os, subprocess, psutil

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if gpu_info.returncode != 0:
    raise RuntimeError("未检测到 GPU！请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> 选择 T4 或 L4 GPU")

gpu_line = gpu_info.stdout.strip()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"检测到 GPU: {gpu_line}")
print(f"系统内存 RAM: {ram_gb:.1f} GB")

if "T4" in gpu_line:
    print("建议：当前为 T4 (16GB, Turing 架构，不支持原生 BF16)，Z-Image-Turbo 推荐选择 gguf_q4，且启动时自动启用 --force-fp16。")
elif "L4" in gpu_line or "A100" in gpu_line:
    print("建议：当前显卡显存 >= 24GB 且支持原生 BF16，Z-Image-Turbo 可直接选择 bf16 或 fp8。")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_WORK_DIR}/loras", exist_ok=True)
    os.makedirs(f"{DRIVE_WORK_DIR}/output", exist_ok=True)
    print(f"已挂载 Google Drive，LoRA 与出图将同步保存在: {DRIVE_WORK_DIR}")


In [ ]:
#@title 2. 安装 ComfyUI、常用插件（Manager + GGUF）与内网穿透工具
REPO_URL = ""  #@param {type:"string"} # 可选：填你的 GitHub 仓库地址，用于自动导入 scripts 与 workflows
INSTALL_MANAGER = True  #@param {type:"boolean"}
INSTALL_GGUF = True  #@param {type:"boolean"}

import os, shutil, subprocess

COMFY_DIR = "/content/ComfyUI"
HELPER_DIR = "/content/comfyui-colab"

!apt-get -y -qq install aria2 > /dev/null

if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}

!pip install -q -r {COMFY_DIR}/requirements.txt

if INSTALL_MANAGER and not os.path.exists(f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"):
    !git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git {COMFY_DIR}/custom_nodes/ComfyUI-Manager

if INSTALL_GGUF and not os.path.exists(f"{COMFY_DIR}/custom_nodes/ComfyUI-GGUF"):
    !git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git {COMFY_DIR}/custom_nodes/ComfyUI-GGUF
    !pip install -q -r {COMFY_DIR}/custom_nodes/ComfyUI-GGUF/requirements.txt

if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
    !dpkg -i /tmp/cloudflared.deb > /dev/null

# 如果填写了你的仓库地址，自动拉取内置工作流和下载脚本
if REPO_URL.strip():
    if not os.path.exists(HELPER_DIR):
        !git clone --depth 1 {REPO_URL.strip()} {HELPER_DIR}
    wf_src = f"{HELPER_DIR}/workflows"
    wf_dst = f"{COMFY_DIR}/user/default/workflows"
    if os.path.isdir(wf_src):
        os.makedirs(wf_dst, exist_ok=True)
        for f in os.listdir(wf_src):
            if f.endswith(".json"):
                shutil.copy2(os.path.join(wf_src, f), os.path.join(wf_dst, f))
        print(f"已导入预设工作流到 {wf_dst}: {os.listdir(wf_dst)}")

# 如果开启了 Google Drive，将 output 和额外 loras 目录做软链接/目录映射
if "USE_GOOGLE_DRIVE" in globals() and USE_GOOGLE_DRIVE:
    out_dir = f"{COMFY_DIR}/output"
    if os.path.isdir(out_dir) and not os.path.islink(out_dir):
        shutil.rmtree(out_dir)
    if not os.path.exists(out_dir):
        os.symlink(f"{DRIVE_WORK_DIR}/output", out_dir)
    drive_lora_link = f"{COMFY_DIR}/models/loras/drive_loras"
    if not os.path.exists(drive_lora_link):
        os.symlink(f"{DRIVE_WORK_DIR}/loras", drive_lora_link)

print("✅ ComfyUI 与插件安装完成！")


In [ ]:
#@title 3. 下载底模（支持 Z-Image-Turbo 预置 & 自定义 SD1.5 / SDXL 大模型）
Z_IMAGE_PRESET = "auto"  #@param ["none", "auto", "gguf_q4", "gguf_q6", "fp8", "bf16"]
CUSTOM_CHECKPOINT_URL = ""  #@param {type:"string"} # 可选：SD 1.5 / SDXL / Illustrious 大模型链接（支持 HuggingFace 或 Civitai）
CIVITAI_API_TOKEN = ""  #@param {type:"string"} # 可选：下载需要登录/NSFW 的 Civitai 模型时填写
HF_TOKEN = ""  #@param {type:"string"}

import os, re, json, subprocess, urllib.request, urllib.parse
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")

def resolve_civitai(url: str, token: str = ""):
    parsed = urllib.parse.urlparse(url)
    if "civitai." not in parsed.netloc.lower():
        return url, None
    if "/api/download/models/" in parsed.path:
        qs = urllib.parse.parse_qs(parsed.query)
        if token and "token" not in qs:
            qs["token"] = [token]
        return urllib.parse.urlunparse(("https", "civitai.com", parsed.path, "", urllib.parse.urlencode(qs, doseq=True), "")), None
    m = re.search(r"/models/(\d+)", parsed.path)
    if not m:
        return url, None
    model_id = m.group(1)
    qs = urllib.parse.parse_qs(parsed.query)
    wanted_vid = qs.get("modelVersionId", [None])[0]
    req = urllib.request.Request(f"https://civitai.com/api/v1/models/{model_id}", headers={"User-Agent": "comfyui-colab/1.0"})
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode())
    versions = data.get("modelVersions", [])
    ver = next((v for v in versions if wanted_vid and str(v["id"]) == str(wanted_vid)), versions[0])
    files = ver.get("files", [])
    primary = next((f for f in files if f.get("primary")), files[0])
    dl_url = f"https://civitai.com/api/download/models/{ver['id']}"
    if token:
        dl_url += f"?token={urllib.parse.quote(token)}"
    print(f"[Civitai] {data.get('name')} | 底模: {ver.get('baseModel')} | 触发词: {ver.get('trainedWords')}")
    return dl_url, primary.get("name")

def dl(url: str, subdir: str, filename: str = None):
    out_dir = COMFY_DIR / "models" / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    resolved, detected = resolve_civitai(url.strip(), CIVITAI_API_TOKEN.strip())
    fname = filename or detected
    if fname and (out_dir / fname).exists() and (out_dir / fname).stat().st_size > 1024 * 1024:
        print(f"[Skip] 已存在: {out_dir / fname}")
        return
    cmd = ["aria2c", "--console-log-level=warn", "-c", "-x", "8", "-s", "8", "-k", "1M", "--auto-file-renaming=false", "-d", str(out_dir)]
    if HF_TOKEN.strip() and "huggingface.co" in resolved:
        cmd += ["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"]
    if fname:
        cmd += ["-o", fname]
    cmd.append(resolved)
    print(f"[Downloading] {fname or resolved} -> models/{subdir}/")
    subprocess.run(cmd, check=True)

preset = Z_IMAGE_PRESET
if preset == "auto":
    gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    preset = "gguf_q4" if "T4" in gpu_name else "bf16"
    print(f"[Auto] 根据显卡 ({gpu_name}) 自动选择预置: {preset}")

PRESETS = {
    "gguf_q4": [
        ("unet", "https://huggingface.co/jayn7/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_K_M.gguf", "z_image_turbo-Q4_K_M.gguf"),
        ("text_encoders", "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf", "Qwen3-4B-Q4_K_M.gguf"),
        ("vae", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "ae.safetensors"),
    ],
    "gguf_q6": [
        ("unet", "https://huggingface.co/jayn7/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q6_K.gguf", "z_image_turbo-Q6_K.gguf"),
        ("text_encoders", "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q6_K.gguf", "Qwen3-4B-Q6_K.gguf"),
        ("vae", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "ae.safetensors"),
    ],
    "fp8": [
        ("diffusion_models", "https://huggingface.co/drbaph/Z-Image-Turbo-FP8/resolve/main/z_image_turbo_fp8_e4m3fn.safetensors", "z_image_turbo_fp8_e4m3fn.safetensors"),
        ("text_encoders", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors", "qwen_3_4b_fp8_mixed.safetensors"),
        ("vae", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "ae.safetensors"),
    ],
    "bf16": [
        ("diffusion_models", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors", "z_image_turbo_bf16.safetensors"),
        ("text_encoders", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors", "qwen_3_4b.safetensors"),
        ("vae", "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors", "ae.safetensors"),
    ],
}

if preset != "none":
    for subdir, url, fname in PRESETS[preset]:
        dl(url, subdir, fname)

if CUSTOM_CHECKPOINT_URL.strip():
    dl(CUSTOM_CHECKPOINT_URL.strip(), "checkpoints")

print("✅ 底模准备完成！")


In [ ]:
#@title 4. 下载 LoRA（可粘贴 Civitai 页面链接或直链，多个用英文逗号分隔）
# 示例：
# https://civitai.red/models/2408517/nsfw-z-image-elegant-oriental-classical-beauty-in-embroidered-lace-hanfu-ensemble
# https://civitai.red/models/136879/r18sunaookami-shiroko-and-shiroko-terror-lora
LORA_URLS = ""  #@param {type:"string"}

urls = [u.strip() for u in LORA_URLS.split(",") if u.strip()]
if not urls:
    print("未填写 LoRA 链接，跳过。需要时填上链接重新运行本单元格，再在网页按 R 刷新即可。")
else:
    for u in urls:
        dl(u, "loras")
    print("✅ LoRA 下载完成！当前 loras 目录文件：", os.listdir("/content/ComfyUI/models/loras"))


In [ ]:
#@title 5. 启动 ComfyUI 网页版（Cloudflared 公网访问链接）
EXTRA_ARGS = ""  #@param {type:"string"} # 可选附加参数，如 --lowvram

import subprocess, threading, time, re, socket

gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
launch_args = ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--dont-print-server"]

# T4 (Turing) 不支持原生 BF16，默认强制 FP16 避免降速或报错
if "T4" in gpu_name and "--force-fp16" not in EXTRA_ARGS:
    launch_args.append("--force-fp16")

if EXTRA_ARGS.strip():
    launch_args += EXTRA_ARGS.strip().split()

print("启动命令:", " ".join(launch_args))

def start_tunnel():
    while True:
        time.sleep(1)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", 8188)) == 0:
                break
    print("\n🚀 ComfyUI 已就绪，正在创建 Cloudflared 隧道...")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("\n" + "=" * 60)
            print("🌐 点击打开 ComfyUI 网页版:", m.group(0))
            print("=" * 60 + "\n")

threading.Thread(target=start_tunnel, daemon=True).start()
subprocess.run(launch_args, cwd="/content/ComfyUI")
